<a href="https://colab.research.google.com/github/Jgriffiths92/Open_E-Dope/blob/master/Kivy_App_To_APK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Open_E-Dope APK Compiler Notebook
This notebook automates system dependency installation, enforces a environment migration to **Python 3.11**, and runs Buildozer compilation for the **Open_E-Dope** application.

### Step 1: Install Ubuntu System Dependencies & Python 3.11
Buildozer requires standard Linux development headers, utilities, and **Java 17** to cross-compile the Android toolchain.

In [ ]:
# Add deadsnakes repository for alternative Python versions
!sudo add-apt-repository ppa:deadsnakes/ppa -y
!sudo apt-get update

# Install essential packages, Java 17, and Python 3.11 system bin targets
!sudo apt-get install -y \
    build-essential \
    uuid-dev \
    zlib1g-dev \
    libncurses5-dev \
    libgdbm-dev \
    libnss3-dev \
    libssl-dev \
    libsqlite3-dev \
    libreadline-dev \
    libffi-dev \
    libbz2-dev \
    liblzma-dev \
    tk-dev \
    libgdbm-compat-dev \
    gawk \
    unzip \
    zip \
    autoconf \
    automake \
    libtool \
    pkg-config \
    cmake \
    git \
    openjdk-17-jdk \
    python3.11 \
    python3.11-dev \
    python3.11-distutils

# Verify Java 17 runtime environmental binding
!java -version

### Step 2: Force Colab Host To Use Python 3.11
This configuration registers Python 3.11 into Colab's internal alternatives matrix and sets it as the primary system interpreter link.

In [ ]:
# Backup the existing default version pointer path mapping
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1
!sudo update-alternatives --set python3 /usr/bin/python3.11

# Verify environment version override success
!python3 --version

### Step 3: Install Buildozer, Cython & Pip
We re-bootstrap pip targets for our new Python 3.11 interpreter path and lock down matching stable Cython builds.

In [ ]:
!curl -sS https://pypa.io | python3
!python3 -m pip install --upgrade buildozer cython==0.29.36 setuptools

### Step 4: Clone Open_E-Dope Repository
This pulls down your codebase structure along with your pre-configured manifest file mappings.

In [ ]:
import os

%cd /content
if not os.path.exists('Open_E-Dope'):
    !git clone https://github.com

%cd /content/Open_E-Dope

### Step 5: Forcibly Sync Python and HostPython Environments
This removes old requirements overrides and explicitly injects dual-pinned parameters (`python3==3.11, hostpython3==3.11`) to prevent Buildozer's automatic environment picker from falling forward to 3.14.

In [ ]:
import re
import os
from pathlib import Path

possible_paths = list(Path('/content').rglob('buildozer.spec'))
if not possible_paths:
    raise FileNotFoundError("buildozer.spec missing. Verify download integrity status logs.")

target_spec = possible_paths[0]
os.chdir(str(target_spec.parent))
content = target_spec.read_text()

# Strip legacy requirements mapping versions cleanly out of the line structure
content = re.sub(r'#?\s*hostpython3\s*==\s*[\d\.]+,?', '', content)
content = re.sub(r'#?\s*python3\s*==\s*[\d\.]+,?', '', content)

# Inject twin constraints to block the automatic 3.14 toolchain fallback safely
if "requirements =" in content:
    content = content.replace("requirements = ", "requirements = python3==3.11, hostpython3==3.11, ")

target_spec.write_text(content)
print("Successfully locked twin requirements constraints mapping parameters!")

### Step 6: Run Buildozer Compilation
This processes the local configuration. Type `y` and hit **Enter** to accept the Android SDK license terms when prompted.

In [ ]:
!buildozer android clean
!buildozer -v android debug

### Step 7: Locate your APK
Once compilation completes successfully, run this cell to verify your output binary inside the `bin/` subfolder.

In [ ]:
!ls -la bin/